In [5]:
# Load API key
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

In [6]:
# Connect to database
db_user = "root"
db_password = "PreronaDhubri123"
db_host = "localhost"
db_name = "jazzy_things"
from langchain_community.utilities.sql_database import SQLDatabase
# db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",sample_rows_in_table_info=1,include_tables=['customers','orders'],custom_table_info={'customers':"customer"})
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")
print(db.dialect)
print(db.get_usable_table_names())
print(db.table_info)

mysql
['customer', 'order_details', 'order_item', 'product']

CREATE TABLE customer (
	id INTEGER NOT NULL AUTO_INCREMENT, 
	first_name VARCHAR(50), 
	last_name VARCHAR(50), 
	email VARCHAR(100), 
	address VARCHAR(100), 
	phone VARCHAR(20), 
	status ENUM('0','1','2'), 
	created_at DATETIME DEFAULT CURRENT_TIMESTAMP, 
	updated_at DATETIME, 
	PRIMARY KEY (id)
)COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB

/*
3 rows from customer table:
id	first_name	last_name	email	address	phone	status	created_at	updated_at
1	Prokritee	Roy Chowdhury	prokriteer@gmail.com	Guwahati, Assam	1234567891	0	2025-11-08 17:10:35	2025-11-15 18:18:02
2	Upashrita	Das	upad@gmail.com	Pune, Maharashtra	9876543219	0	2025-11-08 17:10:35	2025-11-15 18:18:02
3	Prajwalita	Hazarika	prajhaz@gmail.com	Guwahati, Assam	1122334455	0	2025-11-08 17:10:35	2025-11-15 18:18:02
*/


CREATE TABLE order_details (
	id INTEGER NOT NULL AUTO_INCREMENT, 
	customer_id INTEGER NOT NULL, 
	order_date DATETIME DEFAULT CURRENT_T

In [4]:
# Create chain to generate query from NL
from langchain_classic.chains import create_sql_query_chain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  
    temperature=0,
    google_api_key=api_key 
)

custom_prompt = PromptTemplate(
    input_variables=["input", "table_info", "top_k", "dialect"],
    template="""Given an input question, create a syntactically correct {dialect} SQL query.

Use the following table information:
{table_info}

Return top {top_k} results.

IMPORTANT: Output ONLY the SQL query without any prefixes, explanations, or markdown formatting.
Do NOT include 'SQLQuery:', '``````', or any other text.

Question: {input}
"""
)

generate_query = create_sql_query_chain(llm, db, prompt=custom_prompt)

# Generate query
query = generate_query.invoke({"question": "what is price of `Silver Moon` necklace?"})

print("Query:", query)



Query: SELECT price FROM product WHERE name = 'Silver Moon'


In [7]:
# Execute query and get result
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
execute_query = QuerySQLDataBaseTool(db=db)
result = execute_query.invoke(query)
print("Result:", result)

Result: [(Decimal('299.00'),)]


In [8]:
# Frame the query result into natural language answer
from operator import itemgetter 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

answer_prompt = PromptTemplate.from_template(
"""Given the following user question, corresponding SQL query, and the result obtained from the query. Reframe the result obtained from the query into a natural language form and answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)

rephrase_answer = answer_prompt | llm | StrOutputParser()

chain = (
    RunnablePassthrough.assign(query=generate_query).assign(
        result=itemgetter("query") | execute_query
    )
    | rephrase_answer
)

chain.invoke({"question": "How many Beetle earrings are in stock?"})


'There are 15 Beetle earrings in stock.'

In [9]:
# Integrating few-shot prompting for more precise results
examples = [
{
"input": "Show me all available products.",
"query": "SELECT * FROM product WHERE status =1;"
},
{
"input": "What products do you have in the Earrings category?",
"query": "SELECT name, price, stock_quantity FROM product WHERE category = 'Earrings' AND status = 1;"
},
{
"input": "Do you have product Cherry danglers in stock?",
"query": "SELECT name, stock_quantity FROM product WHERE name = ‘Cherry danglers’ AND stock_quantity > 0 AND status = 1;"
},
{
"input": "Show me products under Rs.200.",
"query": "SELECT name, category, price FROM product WHERE price < 200 AND status = 1;"
},
{
"input": "What is the price of the product named 'Rose-white pearl'?",
"query": "SELECT price FROM product WHERE name = 'Rose-white pearl';"
},
{
    "input": "Find my account details, my email is upad@gmail.com.",
    "query": "SELECT first_name, last_name, email, phone, address FROM customer WHERE email = 'upad@gmail.com';"
},
{
    "input": "Is my account active?  My email is upad@gmail.com",
    "query": "SELECT status FROM customer WHERE email = 'upad@gmail.com';"
},
{
    "input": "What is the status of my order? My order number is 3.",
    "query": "SELECT order_status, order_date, total_amount FROM order_details WHERE id = 3;"
},
{
    "input": "How many products are out of stock?",
    "query": "SELECT COUNT(*) as out_of_stock_count FROM product WHERE stock_quantity = 0;"
},
{
    "input": "What categories do you have?",
    "query": "SELECT DISTINCT category FROM product WHERE status = 1;"
}
]

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder,FewShotChatMessagePromptTemplate,PromptTemplate

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}\nSQLQuery:"),
        ("ai", "{query}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    # input_variables=["input","top_k"],
    input_variables=["input"],
)
print(few_shot_prompt.format(input1="How many products are there?"))


Human: Show me all available products.
SQLQuery:
AI: SELECT * FROM product WHERE status =1;
Human: What products do you have in the Earrings category?
SQLQuery:
AI: SELECT name, price, stock_quantity FROM product WHERE category = 'Earrings' AND status = 1;
Human: Do you have product Cherry danglers in stock?
SQLQuery:
AI: SELECT name, stock_quantity FROM product WHERE name = ‘Cherry danglers’ AND stock_quantity > 0 AND status = 1;
Human: Show me products under Rs.200.
SQLQuery:
AI: SELECT name, category, price FROM product WHERE price < 200 AND status = 1;
Human: What is the price of the product named 'Rose-white pearl'?
SQLQuery:
AI: SELECT price FROM product WHERE name = 'Rose-white pearl';
Human: Find my account details, my email is upad@gmail.com.
SQLQuery:
AI: SELECT first_name, last_name, email, phone, address FROM customer WHERE email = 'upad@gmail.com';
Human: Is my account active?  My email is upad@gmail.com
SQLQuery:
AI: SELECT status FROM customer WHERE email = 'upad@gmail.c

In [15]:
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_huggingface import HuggingFaceEmbeddings

vectorstore = Chroma()
vectorstore.delete_collection()
embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2" 
    )
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,
    vectorstore,
    k=2,
    input_keys=["input"],
)
example_selector.select_examples({"input": "how many products fall in the earrings category?"})
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    input_variables=["input","top_k"],
)
print(few_shot_prompt.format(input="how many products fall in the earrings category?"))


d:\projects\customer_support_chatbot\my_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Human: What products do you have in the Earrings category?
SQLQuery:
AI: SELECT name, price, stock_quantity FROM product WHERE category = 'Earrings' AND status = 1;
Human: How many products are out of stock?
SQLQuery:
AI: SELECT COUNT(*) as out_of_stock_count FROM product WHERE stock_quantity = 0;


In [17]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """Given an input question, create a syntactically correct SQL query.

    Use the following table information:
    {table_info}

    IMPORTANT: Output ONLY the SQL query without any prefixes, explanations, or markdown formatting.
    Do NOT include 'SQLQuery:', '``````', or any other text.
    
    Below are a number of examples of questions and their corresponding SQL queries for your reference.
    """),
        few_shot_prompt,
    ("human", "{input}"),
    ]
)
print(final_prompt.format(input="how many products fall in the earrings category?",table_info="some table info"))
generate_query = create_sql_query_chain(llm, db, prompt = final_prompt)
chain = (
RunnablePassthrough.assign(query=generate_query).assign(
    result=itemgetter("query") | execute_query
)
| rephrase_answer
)
chain.invoke({"question": "how many products fall in the earrings category?"})


System: Given an input question, create a syntactically correct SQL query.

    Use the following table information:
    some table info

    IMPORTANT: Output ONLY the SQL query without any prefixes, explanations, or markdown formatting.
    Do NOT include 'SQLQuery:', '``````', or any other text.

    Below are a number of examples of questions and their corresponding SQL queries for your reference.
    
Human: What products do you have in the Earrings category?
SQLQuery:
AI: SELECT name, price, stock_quantity FROM product WHERE category = 'Earrings' AND status = 1;
Human: How many products are out of stock?
SQLQuery:
AI: SELECT COUNT(*) as out_of_stock_count FROM product WHERE stock_quantity = 0;
Human: how many products fall in the earrings category?


'There are 3 products that fall into the earrings category.'